In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Task 1: Write your code here:
path = os.path.join(path, 'Q1_data.csv')

df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  print("Target Distribution:")
  df[target_column].hist()
  plt.show()
check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df_clean = df.drop(columns=['Order_ID'])


In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")
check_missing_values(df_clean)



In [ ]:
df_clean['Weather'] = df_clean['Weather'].fillna('unknown')
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna('unknown')
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna('unknown')

df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

# drop the rows with missing target
df_clean = df_clean.dropna(subset=['Delivery_Time'])



In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    le = LabelEncoder() # 0 1 2 .... like oredering
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))





In [ ]:
# Task 5: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 6: Write your code here:
# not needed when the target is cts in regression tasks

In [ ]:
# Task 1: Write your code here:
# defined in task 5
# X = df_clean.drop("Delivery_Time", axis=1).astype(float)
# y = df['Delivery_Time'].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


loss=[]
model = RandomForestRegressor()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  model.fit(X_train, y_train) # Train
  y_pred = model.predict(X_test) # Predict

  # Calculate metric
  mae = mean_absolute_error(y_test, y_pred)

  loss.append(mae)

avgloss=np.sum(loss)

In [ ]:
avgloss

In [ ]:
df_clean.columns

In [ ]:
# Task 1: Write your code here:
feature_cols=['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs'
       ]
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# then use bar to plot it
plt.barh(feature_importance['feature'], feature_importance['importance'])

In [ ]:
# Task 2: Write your code here:
def plot_target_column(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')
  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)
  plt.show()

plot_target_column(df_clean, "Delivery_Time")

In [ ]:
%pip install kagglehub catboost lightgbm tqdm -q


In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor


models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

all_results = {}

for name in models:
  all_results[name] = {'mae': []}


kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    model.fit(X_train, y_train) # Train
    y_pred = model.predict(X_test) # Predict

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)



# for results comparison
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mse']):.4f}")